In [1]:
import os, time, subprocess, sys
import gradio as gr
from langserve import RemoteRunnable
from langchain_community.document_loaders import PyMuPDFLoader, UnstructuredWordDocumentLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.messages import HumanMessage, AIMessage

PORT = 9017

if sys.platform == "win32":
    os.system(f'for /f "tokens=5" %a in (\'netstat -aon ^| findstr :{PORT}\') do taskkill /f /pid %a > nul 2>&1')
else:
    os.system(f"fuser -k {PORT}/tcp > /dev/null 2>&1")

time.sleep(2)

subprocess.Popen([sys.executable, "server_app.py"])
print("🚀 Initializing Assistant... (15s)")
time.sleep(15)

client = RemoteRunnable(f"http://127.0.0.1:{PORT}/rag_chat/")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = None

def convert_history(history):
    messages = []
    for item in history:
        if isinstance(item, (list, tuple)) and len(item) == 2:
            user_msg, ai_msg = item
            if user_msg:
                messages.append(HumanMessage(content=user_msg))
            if ai_msg:
                messages.append(AIMessage(content=ai_msg))
    return messages

def process_files(files):
    global vectorstore
    if not files:
        return "No files uploaded."

    file_list = files if isinstance(files, list) else [files]
    all_docs = []

    try:
        for file in file_list:
            if file.name.lower().endswith('.pdf'):
                loader = PyMuPDFLoader(file.name)
            elif file.name.lower().endswith('.docx'):
                loader = UnstructuredWordDocumentLoader(file.name)
            else:
                continue

            pages = loader.load()
            for i, page in enumerate(pages):
                page.metadata["page"] = i + 1
                page.page_content = f"[[PAGE {i+1}]] {page.page_content}"
                all_docs.append(page)

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=900,
            chunk_overlap=250
        )
        splits = splitter.split_documents(all_docs)
        vectorstore = Chroma.from_documents(splits, embeddings)
        return f"✅ READY: {len(all_docs)} pages indexed."
    except Exception as e:
        return f"❌ Ingestion Error: {str(e)}"

def chat_engine(message, history):
    global vectorstore
    history = history or []

    if vectorstore is None:
        if any(greet in message.lower() for greet in ["hello", "hi", "hey"]):
            return "Hello! Please upload a document first."
        return "⚠️ Please upload documents first."

    try:
        history = history[-4:]
        docs = vectorstore.similarity_search(message, k=8)
        docs.sort(key=lambda x: x.metadata.get('page', 0))
        context_text = "\n\n".join([d.page_content for d in docs])
        langchain_history = convert_history(history)
        response = client.invoke({
            "question": message,
            "context": context_text,
            "chat_history": langchain_history
        })

        return response

    except Exception as e:
        return f"❌ AI Error: {str(e)}"

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🚀 Smart Contract Summary & Q&A Assistant")
    gr.Markdown("Upload → Ask → Get Cited Answers")

    with gr.Row():
        file_in = gr.File(label="Upload PDF/DOCX", file_count="multiple")
        status = gr.Textbox(label="System Status", interactive=False)

    file_in.change(process_files, file_in, status)
    gr.ChatInterface(fn=chat_engine)

demo.launch(inline=True)

b:\RAG Assigment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 Initializing Assistant... (15s)


C:\Users\fares\AppData\Local\Temp\ipykernel_41716\1941777643.py:101: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
